In [1]:
import pandas as pd
import json
import numpy as np

In [2]:
df = pd.read_json('E:\\DATASCIENCE PORTFOLIO\\Practice_Projects\\Instacart Data Transformation\\Dataset\\case.json')

In [3]:
df.head()

,EnqueuedTimeUtc,EventName,Payload
0,2021-09-05 08:04:08 UTC,DynamicPrice_Result,"{""provider"":""ApplyDynamicPriceRange"",""offerId""..."
1,2021-08-18 11:43:23 UTC,DynamicPrice_Result,"{""provider"":""ApplyDynamicPricePerOption"",""offe..."
2,2021-09-05 09:04:04 UTC,DynamicPrice_Result,"{""provider"":""ApplyDynamicPriceRange"",""offerId""..."
3,2021-08-25 05:02:55 UTC,CurateOffer_Result,"[{""curationProvider"":""ByPrice"",""offerId"":""149f..."
4,2021-09-05 08:03:28 UTC,DynamicPrice_Result,"{""provider"":""ApplyDynamicPriceRange"",""offerId""..."


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 37 entries, 0 to 36
Data columns (total 3 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   EnqueuedTimeUtc  37 non-null     object
 1   EventName        37 non-null     object
 2   Payload          37 non-null     object
dtypes: object(3)
memory usage: 1020.0+ bytes


In [5]:
brazil_tz = pd.to_datetime(df['EnqueuedTimeUtc']).dt.tz_convert('America/Sao_Paulo')
df['EnqueuedTimeUtc'] = brazil_tz.dt.strftime('%d/%m/%y')

In [6]:
curate_offer = df[df['EventName'] == 'CurateOffer_Result'].reset_index(drop = True)
curate_offer.head()

,EnqueuedTimeUtc,EventName,Payload
0,25/08/21,CurateOffer_Result,"[{""curationProvider"":""ByPrice"",""offerId"":""149f..."
1,25/08/21,CurateOffer_Result,"[{""curationProvider"":""ByPrice"",""offerId"":""c99a..."
2,25/08/21,CurateOffer_Result,"[{""curationProvider"":""ByPrice"",""offerId"":""d668..."
3,25/08/21,CurateOffer_Result,"[{""curationProvider"":""ByPrice"",""offerId"":""0a06..."
4,25/08/21,CurateOffer_Result,"[{""curationProvider"":""ByPrice"",""offerId"":""135d..."


In [7]:
co_list = []
for idx, row in curate_offer.iterrows():
    data = json.loads(row['Payload'])
    payload_data = pd.json_normalize(data,
                            record_path = ['options'],
                            meta = ['curationProvider', 'offerId', 'dealerId'])
    payload_data['EnqueuedTimeSP'] = row['EnqueuedTimeUtc']

    co_list.append(payload_data)
    

CuratedOfferOptions = pd.concat(co_list, axis= 0).reset_index(drop = True)


column_order = {'curationProvider':'CurationProvider', 
                'offerId':'OfferId', 
                'dealerId':'DealerId', 
                'uniqueOptionId':'UniqueOptionId', 
                'optionId':'OptionId',
                'isMobileDealer':'IsMobileDealer', 
                'isOpen':'IsOpen', 
                'eta':'Eta',
                'chamaScore':'ChamaScore',
                'productBrand':'ProductBrand',
                'isWinner':'IsWinner', 
                'minimumPrice':'MinimumPrice',
                'maximumPrice':'MaximumPrice', 
                'dynamicPrice':'DynamicPrice', 
                'finalPrice':'FinalPrice', 
                'defeatPrimaryReason':'DefeatPrimaryReason', 
                'defeatReasons':'DefeatReasons', 
                'EnqueuedTimeSP':'EnqueuedTimeSP'}

CuratedOfferOptions = CuratedOfferOptions[column_order.keys()].rename(columns = column_order)
CuratedOfferOptions = CuratedOfferOptions.fillna('')
ob_col = CuratedOfferOptions.select_dtypes(include=['object']).columns
CuratedOfferOptions[ob_col] = CuratedOfferOptions[ob_col].astype(str)


In [8]:
CuratedOfferOptions.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 40 entries, 0 to 39
Data columns (total 18 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   CurationProvider     40 non-null     object 
 1   OfferId              40 non-null     object 
 2   DealerId             40 non-null     object 
 3   UniqueOptionId       40 non-null     object 
 4   OptionId             40 non-null     object 
 5   IsMobileDealer       40 non-null     bool   
 6   IsOpen               40 non-null     bool   
 7   Eta                  40 non-null     object 
 8   ChamaScore           40 non-null     float64
 9   ProductBrand         40 non-null     object 
 10  IsWinner             40 non-null     bool   
 11  MinimumPrice         40 non-null     float64
 12  MaximumPrice         40 non-null     float64
 13  DynamicPrice         40 non-null     float64
 14  FinalPrice           40 non-null     float64
 15  DefeatPrimaryReason  40 non-null     objec

In [9]:
df.head()

,EnqueuedTimeUtc,EventName,Payload
0,05/09/21,DynamicPrice_Result,"{""provider"":""ApplyDynamicPriceRange"",""offerId""..."
1,18/08/21,DynamicPrice_Result,"{""provider"":""ApplyDynamicPricePerOption"",""offe..."
2,05/09/21,DynamicPrice_Result,"{""provider"":""ApplyDynamicPriceRange"",""offerId""..."
3,25/08/21,CurateOffer_Result,"[{""curationProvider"":""ByPrice"",""offerId"":""149f..."
4,05/09/21,DynamicPrice_Result,"{""provider"":""ApplyDynamicPriceRange"",""offerId""..."


In [10]:
dynamic_price = df[df['EventName'] == 'DynamicPrice_Result'].reset_index(drop = True)
dynamic_price['EnqueuedTimeUtc'].value_counts()

EnqueuedTimeUtc
05/09/21    25
18/08/21     7
Name: count, dtype: int64

In [11]:
dpoption_list = []
dprrange_list = []
for idx, row in dynamic_price.iterrows():
    
    data = json.loads(row['Payload'])
    
    if data['provider'] == 'ApplyDynamicPriceRange':
        
        dprange = pd.json_normalize(data)
        dprange['EnqueuedTimeSP'] = row['EnqueuedTimeUtc']
        dprrange_list.append(dprange)
        
    if data['provider'] == 'ApplyDynamicPricePerOption':

        dpoption = pd.json_normalize(data,
                                    record_path=['algorithmOutput'],
                                    meta = ['provider','offerId'])
        dpoption['EnqueuedTimeSP'] = row['EnqueuedTimeUtc']
        dpoption_list.append(dpoption)

    else:
        pass



DynamicPriceOption = pd.concat(dpoption_list, axis = 0, ignore_index = True)
DynamicPriceRange = pd.concat(dprrange_list,axis = 0, ignore_index= True)


dpoption_col = {'provider':'Provider', 
                'offerId':'OfferId', 
                'uniqueOptionId':'UniqueOptionId', 
                'bestPrice':'BestPrice', 
                'EnqueuedTimeSP':'EnqueuedTimeSP'}

DynamicPriceOption = DynamicPriceOption[dpoption_col.keys()].rename(columns = dpoption_col)
DynamicPriceOption.columns



dprange_col = {'provider':'Provider', 
               'offerId':'OfferId', 
                'algorithmOutput.min_global':'MinimumGlobal',
               'algorithmOutput.min_recommended':'MaximumRecommended', 
               'algorithmOutput.max_recommended' : 'MaximumRecommended',
               'algorithmOutput.differenceMinRecommendMinTheory' : 'DifferenceMinRecommendMinTheory', 
               'EnqueuedTimeSP':'EnqueuedTimeSP'}

DynamicPriceRange = DynamicPriceRange[dprange_col.keys()].rename(columns = dprange_col)





In [12]:
DynamicPriceRange.head()

,Provider,OfferId,MinimumGlobal,MaximumRecommended,MaximumRecommended,DifferenceMinRecommendMinTheory,EnqueuedTimeSP
0,ApplyDynamicPriceRange,a6611d55-9624-4381-8cdd-323ee3689241,85.00,87.20,97.65,2.2,05/09/21
1,ApplyDynamicPriceRange,b8c636fa-8241-47dc-ac40-bdf438a04d9c,85.00,87.20,97.65,2.2,05/09/21
2,ApplyDynamicPriceRange,3d32f7fb-396d-4d3f-b673-dea1f7dc41b7,85.00,87.20,97.65,2.2,05/09/21
3,ApplyDynamicPriceRange,329194f3-95a4-45ef-b3d0-2796f74ce2a0,85.00,87.20,97.65,2.2,05/09/21
4,ApplyDynamicPriceRange,fdcfde5c-113d-4a59-9ae0-8bc31e2943d8,87.35,89.25,99.95,1.9,05/09/21


In [13]:
DynamicPriceOption.head()

,Provider,OfferId,UniqueOptionId,BestPrice,EnqueuedTimeSP
0,ApplyDynamicPricePerOption,56e0702c-0218-4626-8d3d-ae9d54b4503b,b0e296a9-0590-f0e0-8211-243a2ededb12,92.45,18/08/21
1,ApplyDynamicPricePerOption,56e0702c-0218-4626-8d3d-ae9d54b4503b,d6562c24-0b37-5fb4-8275-65b7b8b47b87,92.45,18/08/21
2,ApplyDynamicPricePerOption,56e0702c-0218-4626-8d3d-ae9d54b4503b,8d0f9262-f543-d0c8-a869-33985ae3ecda,92.45,18/08/21
3,ApplyDynamicPricePerOption,56e0702c-0218-4626-8d3d-ae9d54b4503b,151e59ac-761a-96f5-d2b9-882037a9fd28,94.60,18/08/21
4,ApplyDynamicPricePerOption,56e0702c-0218-4626-8d3d-ae9d54b4503b,3cd346f4-d297-7568-2e50-d43a8e2fd0a9,94.60,18/08/21
